# 02 · Cleaning and Train/Test split

**Goal.** Prepare the data for modelling: drop useless columns, separate features from the target, and create a train set and a held-out test set.


In [1]:
# Imports, random seed (for reproducibility) and file paths
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
RAW_PATH = Path("../data/raw/diabetes_dataset.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load the raw data

In [3]:
# Load the raw dataset
df = pd.read_csv("../data/raw/diabetes_dataset.csv")
print(df.shape)
df.head()

(500, 10)


,Patient_ID,Age,Gender,BMI,Glucose_Level,Blood_Pressure,Insulin,Physical_Activity,Family_History,Diabetes_Outcome
0,P0001,58,F,36.4,151,102,98.5,86,Yes,1
1,P0002,71,M,28.3,185,80,294.3,29,No,0
2,P0003,48,M,27.1,172,82,186.5,108,Yes,1
3,P0004,34,F,24.0,86,139,181.0,46,No,0
4,P0005,62,M,19.2,83,106,228.2,86,Yes,0


## 2. Cleaning
`Patient_ID` is only an identifier, so it carries no predictive information and is removed. Duplicated rows are also dropped (none were found in the EDA).

In [4]:
# Remove the identifier and any duplicates; check that no value is missing
df_clean = df.drop(columns=["Patient_ID"]).drop_duplicates().reset_index(drop=True)
print("Avant :", df.shape, "| Après :", df_clean.shape)
print("Valeurs manquantes :", df_clean.isna().sum().sum())

Avant : (500, 10) | Après : (500, 9)
Valeurs manquantes : 0


## 3. Separate features (X) and target (y)
`X` holds the 8 input variables and `y` holds `Diabetes_Outcome`.

In [5]:
# X = all columns except the target, y = the target column
X = df_clean.drop(columns="Diabetes_Outcome")
y = df_clean["Diabetes_Outcome"]
print("X :", X.shape, "| y :", y.shape)
X.head()

X : (500, 8) | y : (500,)


,Age,Gender,BMI,Glucose_Level,Blood_Pressure,Insulin,Physical_Activity,Family_History
0,58,F,36.4,151,102,98.5,86,Yes
1,71,M,28.3,185,80,294.3,29,No
2,48,M,27.1,172,82,186.5,108,Yes
3,34,F,24.0,86,139,181.0,46,No
4,62,M,19.2,83,106,228.2,86,Yes


## 4. Train/Test split (80/20, stratified)
- `test_size=0.20`: 400 patients for training, 100 for the final test.
- `stratify=y`: keeps the same proportion of diabetic patients in both sets.
- `random_state`: makes the split reproducible.

The test set is **set aside** and used only once, at the very end of notebook 03.

In [7]:
# Stratified split; the printed proportions of class 1 should be almost identical in all three sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train :", X_train.shape, "| Test :", X_test.shape)
print(f"Proportion de la classe 1 -> global: {y.mean():.3f} | train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

Train : (400, 8) | Test : (100, 8)
Proportion de la classe 1 -> global: 0.542 | train: 0.542 | test: 0.540


## 5. Save the datasets
Saving the split guarantees that notebook 03 uses exactly the same train and test rows.

In [8]:
# Re-attach the target to each set and save them as CSV files
train = X_train.assign(Diabetes_Outcome=y_train)
test = X_test.assign(Diabetes_Outcome=y_test)

train.to_csv(PROCESSED_DIR / "train.csv", index=False)
test.to_csv(PROCESSED_DIR / "test.csv", index=False)
print("Fichiers sauvegardés dans", PROCESSED_DIR)

Fichiers sauvegardés dans ..\data\processed
